# 02 · Consultas, arrays, agregação e paginação

**Pré-requisito:** notebooks 00 e 01.

**Nesta etapa:** perguntar coisas ao banco. No fim, o **Exercício 2** (itens a até e).

## Conexão com o seu MongoDB

A célula abaixo lê o endereço do banco (`MONGO_URI`) do arquivo **`.env`** da pasta, criado no notebook 00, e testa a conexão. Se der erro, confira se o MongoDB está rodando (notebook 00, Etapa 4).

In [1]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv()                                   # lê o arquivo .env da pasta
client = MongoClient(os.environ["MONGO_URI"])
client.admin.command("ping")
print("Conectado ao MongoDB")

Conectado ao MongoDB


Para todos terem o mesmo resultado, vamos recriar a coleção com os 8 alunos da aula:

In [2]:
ALUNOS = [
    {"nome": "Ana",     "idade": 22, "curso": "Computação"},
    {"nome": "Bruno",   "idade": 19, "curso": "Medicina"},
    {"nome": "Carla",   "idade": 24, "curso": "Direito"},
    {"nome": "Matheus", "idade": 21, "curso": "Computação", "disciplinas": ["Python"]},
    {"nome": "Mariana", "idade": 26, "curso": "Engenharia"},
    {"nome": "Pedro",   "idade": 20, "curso": "Medicina"},
    {"nome": "Juliana", "idade": 23, "curso": "Ciência de Dados"},
    {"nome": "Lucas",   "idade": 18, "curso": "Direito"},
]

alunos = client["escola"]["alunos"]
alunos.delete_many({})
alunos.insert_many(ALUNOS)
print(alunos.count_documents({}), "alunos")

8 alunos



## 1. Uma consulta também é um documento

| SQL | MongoDB |
|---|---|
| `WHERE idade > 21` | `{"idade": {"$gt": 21}}` |

### Operadores de comparação

| Operador | Significado |
|---|---|
| `$eq` / `$ne` | igual / diferente |
| `$gt` / `$gte` | maior / maior ou igual |
| `$lt` / `$lte` | menor / menor ou igual |
| `$in` / `$nin` | está / não está na lista |

In [3]:
# projeção usada nos exemplos: mostra só nome, idade e curso
campos = {"_id": 0, "nome": 1, "idade": 1, "curso": 1}

In [4]:
print("Idade maior que 21:")
for a in alunos.find({"idade": {"$gt": 21}}, campos):
    print(a)

Idade maior que 21:
{'nome': 'Ana', 'idade': 22, 'curso': 'Computação'}
{'nome': 'Carla', 'idade': 24, 'curso': 'Direito'}
{'nome': 'Mariana', 'idade': 26, 'curso': 'Engenharia'}
{'nome': 'Juliana', 'idade': 23, 'curso': 'Ciência de Dados'}


In [5]:
print("Idade entre 20 e 23:")
for a in alunos.find({"idade": {"$gte": 20, "$lte": 23}}, campos):   # dois operadores no mesmo campo = E
    print(a)

Idade entre 20 e 23:
{'nome': 'Ana', 'idade': 22, 'curso': 'Computação'}
{'nome': 'Matheus', 'idade': 21, 'curso': 'Computação'}
{'nome': 'Pedro', 'idade': 20, 'curso': 'Medicina'}
{'nome': 'Juliana', 'idade': 23, 'curso': 'Ciência de Dados'}


In [6]:
print("Curso em Direito ou Medicina:")
for a in alunos.find({"curso": {"$in": ["Direito", "Medicina"]}}, campos):
    print(a)

Curso em Direito ou Medicina:
{'nome': 'Bruno', 'idade': 19, 'curso': 'Medicina'}
{'nome': 'Carla', 'idade': 24, 'curso': 'Direito'}
{'nome': 'Pedro', 'idade': 20, 'curso': 'Medicina'}
{'nome': 'Lucas', 'idade': 18, 'curso': 'Direito'}


In [7]:
print("Curso fora de Direito e Medicina:")
for a in alunos.find({"curso": {"$nin": ["Direito", "Medicina"]}}, campos):
    print(a)

Curso fora de Direito e Medicina:
{'nome': 'Ana', 'idade': 22, 'curso': 'Computação'}
{'nome': 'Matheus', 'idade': 21, 'curso': 'Computação'}
{'nome': 'Mariana', 'idade': 26, 'curso': 'Engenharia'}
{'nome': 'Juliana', 'idade': 23, 'curso': 'Ciência de Dados'}


### Operadores lógicos

| Operador | Significado |
|---|---|
| `$and` | todas as condições |
| `$or` | pelo menos uma |
| `$nor` | nenhuma delas |
| `$not` | nega uma condição |

In [8]:
condicoes = [{"idade": {"$lt": 21}}, {"curso": "Direito"}]

In [9]:
print("$and (menor de 21 E de Direito):")
for a in alunos.find({"$and": condicoes}, campos):
    print(a)

$and (menor de 21 E de Direito):
{'nome': 'Lucas', 'idade': 18, 'curso': 'Direito'}


In [10]:
print("$or (menor de 21 OU de Direito):")
for a in alunos.find({"$or": condicoes}, campos):
    print(a)

$or (menor de 21 OU de Direito):
{'nome': 'Bruno', 'idade': 19, 'curso': 'Medicina'}
{'nome': 'Carla', 'idade': 24, 'curso': 'Direito'}
{'nome': 'Pedro', 'idade': 20, 'curso': 'Medicina'}
{'nome': 'Lucas', 'idade': 18, 'curso': 'Direito'}


In [11]:
print("$nor (nem menor de 21, nem de Direito):")
for a in alunos.find({"$nor": condicoes}, campos):
    print(a)

$nor (nem menor de 21, nem de Direito):
{'nome': 'Ana', 'idade': 22, 'curso': 'Computação'}
{'nome': 'Matheus', 'idade': 21, 'curso': 'Computação'}
{'nome': 'Mariana', 'idade': 26, 'curso': 'Engenharia'}
{'nome': 'Juliana', 'idade': 23, 'curso': 'Ciência de Dados'}


In [12]:
print("$not (idade NÃO maior que 21):")
for a in alunos.find({"idade": {"$not": {"$gt": 21}}}, campos):
    print(a)

$not (idade NÃO maior que 21):
{'nome': 'Bruno', 'idade': 19, 'curso': 'Medicina'}
{'nome': 'Matheus', 'idade': 21, 'curso': 'Computação'}
{'nome': 'Pedro', 'idade': 20, 'curso': 'Medicina'}
{'nome': 'Lucas', 'idade': 18, 'curso': 'Direito'}


### Existência, tipo e texto

| Operador | Significado |
|---|---|
| `$exists` | o campo existe neste documento? |
| `$type` | o campo é de um certo tipo? |
| `$regex` | o texto segue um padrão? (`^M` = começa com M) |

In [13]:
print("Quem tem o campo disciplinas:")
for a in alunos.find({"disciplinas": {"$exists": True}}, campos):
    print(a)

Quem tem o campo disciplinas:
{'nome': 'Matheus', 'idade': 21, 'curso': 'Computação'}


In [14]:
print("Quem tem idade do tipo inteiro:")
for a in alunos.find({"idade": {"$type": "int"}}, campos):
    print(a)

Quem tem idade do tipo inteiro:
{'nome': 'Ana', 'idade': 22, 'curso': 'Computação'}
{'nome': 'Bruno', 'idade': 19, 'curso': 'Medicina'}
{'nome': 'Carla', 'idade': 24, 'curso': 'Direito'}
{'nome': 'Matheus', 'idade': 21, 'curso': 'Computação'}
{'nome': 'Mariana', 'idade': 26, 'curso': 'Engenharia'}
{'nome': 'Pedro', 'idade': 20, 'curso': 'Medicina'}
{'nome': 'Juliana', 'idade': 23, 'curso': 'Ciência de Dados'}
{'nome': 'Lucas', 'idade': 18, 'curso': 'Direito'}


In [15]:
print("Nome começa com M:")
for a in alunos.find({"nome": {"$regex": "^M"}}, campos):
    print(a)

Nome começa com M:
{'nome': 'Matheus', 'idade': 21, 'curso': 'Computação'}
{'nome': 'Mariana', 'idade': 26, 'curso': 'Engenharia'}


In [16]:
print("Nome começa com m, ignorando maiúsculas/minúsculas:")
for a in alunos.find({"nome": {"$regex": "^m", "$options": "i"}}, campos):   # "i" ignora maiúsculas/minúsculas
    print(a)

Nome começa com m, ignorando maiúsculas/minúsculas:
{'nome': 'Matheus', 'idade': 21, 'curso': 'Computação'}
{'nome': 'Mariana', 'idade': 26, 'curso': 'Engenharia'}


## 2. Atualizando campos e listas

| Operador | O que faz |
|---|---|
| `$set` / `$unset` | define / remove um campo |
| `$inc` / `$mul` | soma / multiplica |
| `$rename` | renomeia um campo |
| `$min` / `$max` | só troca se o novo valor for menor / maior |
| `$currentDate` | grava a data de agora |
| `$push` | adiciona ao fim da lista (mesmo se já existir) |
| `$addToSet` | adiciona só se ainda não existir |
| `$each` | vários itens de uma vez (com `$push` ou `$addToSet`) |
| `$pop` | remove o último (`1`) ou o primeiro (`-1`) |
| `$pull` | remove um valor específico |

In [17]:
m = {"nome": "Matheus"}
so_disciplinas = {"_id": 0, "disciplinas": 1}

In [18]:
alunos.update_one(m, {"$push": {"disciplinas": "SQL"}})
print("$push SQL:        ", alunos.find_one(m, so_disciplinas))

$push SQL:         {'disciplinas': ['Python', 'SQL']}


In [19]:
alunos.update_one(m, {"$addToSet": {"disciplinas": "SQL"}})          # já existe: nada muda
print("$addToSet SQL:    ", alunos.find_one(m, so_disciplinas))

$addToSet SQL:     {'disciplinas': ['Python', 'SQL']}


In [20]:
alunos.update_one(m, {"$push": {"disciplinas": {"$each": ["Estatística", "IA"]}}})
print("$push $each:      ", alunos.find_one(m, so_disciplinas))

$push $each:       {'disciplinas': ['Python', 'SQL', 'Estatística', 'IA']}


In [21]:
alunos.update_one(m, {"$pop": {"disciplinas": -1}})                  # tira o primeiro
print("$pop -1:          ", alunos.find_one(m, so_disciplinas))

$pop -1:           {'disciplinas': ['SQL', 'Estatística', 'IA']}


In [22]:
alunos.update_one(m, {"$pull": {"disciplinas": "SQL"}})
print("$pull SQL:        ", alunos.find_one(m, so_disciplinas))

$pull SQL:         {'disciplinas': ['Estatística', 'IA']}


In [23]:
alunos.update_one({"nome": "Pedro"}, {"$inc": {"idade": 1}})            # aniversário
alunos.update_many({}, {"$currentDate": {"atualizado_em": True}})
print(alunos.find_one({"nome": "Pedro"}, {"_id": 0}))

{'nome': 'Pedro', 'idade': 21, 'curso': 'Medicina', 'atualizado_em': datetime.datetime(2026, 11, 24, 0, 54, 9, 816000)}


In [24]:
# desfaz para os próximos exemplos
alunos.update_one({"nome": "Pedro"}, {"$inc": {"idade": -1}})
alunos.update_many({}, {"$unset": {"atualizado_em": ""}})

UpdateResult({'n': 8, 'nModified': 8, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

---
## 3. Agregação: a esteira de fábrica

Um **pipeline** é uma lista de estágios. Os documentos entram no primeiro, e cada estágio trabalha sobre o resultado do anterior.

| Estágio | Em SQL | O que faz |
|---|---|---|
| `$match` | `WHERE` | filtra |
| `$group` | `GROUP BY` | agrupa e calcula `$sum`, `$avg`, `$min`, `$max` |
| `$project` | `SELECT` | escolhe e cria campos |
| `$sort` | `ORDER BY` | ordena (`1` cresc., `-1` decresc.) |
| `$skip` / `$limit` | `OFFSET` / `LIMIT` | pula / limita |

No `$group`, o campo de agrupamento vai em `_id`, e **o nome de um campo precisa de `$` na frente** (`"$curso"`) para virar o *valor* do campo.

In [25]:
pipeline = [
    {"$match": {"idade": {"$gte": 20}}},
    {"$group": {"_id": "$curso", "total": {"$sum": 1}, "media": {"$avg": "$idade"}}},
    {"$sort": {"media": -1}},
]
for linha in alunos.aggregate(pipeline):
    print(linha)

{'_id': 'Engenharia', 'total': 1, 'media': 26.0}
{'_id': 'Direito', 'total': 1, 'media': 24.0}
{'_id': 'Ciência de Dados', 'total': 1, 'media': 23.0}
{'_id': 'Computação', 'total': 2, 'media': 21.5}
{'_id': 'Medicina', 'total': 1, 'media': 20.0}


In [26]:
# $project: renomear, esconder e calcular campos
for linha in alunos.aggregate([
    {"$group": {"_id": "$curso", "media": {"$avg": "$idade"}}},
    {"$project": {"_id": 0, "curso": "$_id", "media": {"$round": ["$media", 1]}}},
]):
    print(linha)

{'curso': 'Engenharia', 'media': 26.0}
{'curso': 'Medicina', 'media': 19.5}
{'curso': 'Computação', 'media': 21.5}
{'curso': 'Direito', 'media': 21.0}
{'curso': 'Ciência de Dados', 'media': 23.0}


---
## 4. Paginação com `skip` + `limit`

Página *p* com *n* itens: pule `(p − 1) × n`, pegue `n`.

In [27]:
por_pagina = 3
total = alunos.count_documents({})
paginas = (total + por_pagina - 1) // por_pagina     # divisão arredondando para cima

for p in range(1, paginas + 1):
    print(f"--- Página {p} ---")
    for a in alunos.find({}, {"_id": 0, "nome": 1}).sort("nome", 1).skip((p - 1) * por_pagina).limit(por_pagina):
        print(a["nome"])

--- Página 1 ---
Ana
Bruno
Carla
--- Página 2 ---
Juliana
Lucas
Mariana
--- Página 3 ---
Matheus
Pedro


---
## Agora vamos fazer o Exercício 2 · Consultas da secretaria

**Você vai usar:** `aggregate`, `$group`, `$sort`, `$or`/`$and`, `$regex`, `$push` + `$each`, `$addToSet`, `$pop`, `skip`/`limit`, `$multiply`.

Antes de começar, rode a célula abaixo para garantir os 8 alunos originais.

In [28]:
alunos.delete_many({})
alunos.insert_many([dict(a) for a in ALUNOS])
print(alunos.count_documents({}), "alunos prontos")

8 alunos prontos


**a) Estatísticas por curso.** Para cada curso: idade **média**, **mínima**, **máxima** e **total** de alunos, da maior média para a menor.

*Resultado esperado (ordem):* Engenharia (26), Ciência de Dados (23), Computação (21,5), Direito (21), Medicina (19,5).

In [ ]:
for curso in alunos.aggregate([
    {"$group": {
        "_id": "$curso",
        "media": {"$avg": "$idade"},
        "minima": {"$min": "$idade"},
        "maxima": {"$max": "$idade"},
        "total": {"$sum": 1}
    }},
    {"$project": {
        "_id": 0,
        "curso": "$_id",
        "media": {"$round": ["$media", 1]},
        "minima": 1,
        "maxima": 1,
        "total": 1
    }},
    {"$sort": {"media": -1}}
]):
    print(curso)

{'minima': 26, 'maxima': 26, 'total': 1, 'curso': 'Engenharia', 'media': 26.0}
{'minima': 23, 'maxima': 23, 'total': 1, 'curso': 'Ciência de Dados', 'media': 23.0}
{'minima': 21, 'maxima': 22, 'total': 2, 'curso': 'Computação', 'media': 21.5}
{'minima': 18, 'maxima': 24, 'total': 2, 'curso': 'Direito', 'media': 21.0}
{'minima': 19, 'maxima': 20, 'total': 2, 'curso': 'Medicina', 'media': 19.5}


**b) Filtro combinado.** Alunos que (têm idade **entre 20 e 25** **e** curso **diferente de Medicina**) **ou** cujo nome começa com **“M”**.

*Resultado esperado:* Ana, Carla, Matheus, Mariana, Juliana.

In [37]:
for aluno in alunos.aggregate([
    {"$match": {
        "$or": [
            {"$and": [
                {"idade": {"$gte": 20, "$lte": 25}},
                {"curso": {"$ne": "Medicina"}}
            ]},
            {"nome": {"$regex": "^M"}}
        ]
    }},
    {"$project": campos},
    {"$sort": {"nome": 1}}
]):
    print(aluno)

{'nome': 'Ana', 'idade': 22, 'curso': 'Computação'}
{'nome': 'Carla', 'idade': 24, 'curso': 'Direito'}
{'nome': 'Juliana', 'idade': 23, 'curso': 'Ciência de Dados'}
{'nome': 'Mariana', 'idade': 26, 'curso': 'Engenharia'}
{'nome': 'Matheus', 'idade': 21, 'curso': 'Computação'}


**c) Arrays.**
1. Adicione `["Banco de Dados", "Estatística"]` a **todos** os alunos (`$push` + `$each`).
2. Adicione `"Machine Learning"` só onde ainda **não** existir (`$addToSet`).
3. Remova a **primeira** disciplina de cada lista (`$pop`).
4. Mostre `nome` e `disciplinas`.

*Dica:* `update_many({}, ...)` aplica a todos. Para o Matheus, o resultado esperado é `["Banco de Dados", "Estatística", "Machine Learning"]`.

In [ ]:
alunos.update_many(
    {},
    {"$push": {"disciplinas": {"$each": ["Banco de Dados", "Estatística"]}}}
)

alunos.update_many(
    {},
    {"$addToSet": {"disciplinas": "Machine Learning"}}
)

alunos.update_many(
    {},
    {"$pop": {"disciplinas": -1}}
)

for aluno in alunos.find(
    {},
    {"_id": 0, "nome": 1, "disciplinas": 1}
):
    print(aluno)

{'nome': 'Ana', 'disciplinas': ['Estatística', 'Machine Learning']}
{'nome': 'Bruno', 'disciplinas': ['Estatística', 'Machine Learning']}
{'nome': 'Carla', 'disciplinas': ['Estatística', 'Machine Learning']}
{'nome': 'Matheus', 'disciplinas': ['Banco de Dados', 'Estatística', 'Machine Learning']}
{'nome': 'Mariana', 'disciplinas': ['Estatística', 'Machine Learning']}
{'nome': 'Pedro', 'disciplinas': ['Estatística', 'Machine Learning']}
{'nome': 'Juliana', 'disciplinas': ['Estatística', 'Machine Learning']}
{'nome': 'Lucas', 'disciplinas': ['Estatística', 'Machine Learning']}


**d) Paginação.** Mostre **3 alunos por página**, só `nome` e `curso`, ordenados pela **idade crescente**, com o número da página antes de cada grupo.

*Resultado esperado na página 1:* Lucas, Bruno, Pedro.

In [36]:
for p in range(1, paginas + 1):
    print(f"--- Página {p} ---")
    for a in alunos.find({}, {"_id": 0, "nome": 1, "curso": 1}).sort("idade", 1).skip((p - 1) * por_pagina).limit(por_pagina):
        print(a["nome"])

--- Página 1 ---
Lucas
Bruno
Pedro
--- Página 2 ---
Matheus
Ana
Juliana
--- Página 3 ---
Carla
Mariana


**e) Campo calculado.** Para cursos cujo nome contém a letra **“a”**, mostre a média de idade e um campo `idade_media_meses` (média × 12, use `$multiply`), do **menor** para o **maior**.

*Resultado esperado:* Medicina (234), Computação (258), Ciência de Dados (276), Engenharia (312). Direito não tem “a”!

In [ ]:
for curso in alunos.aggregate([
    {"$match": {"curso": {"$regex": "a"}}},
    {"$group": {
        "_id": "$curso",
        "media": {"$avg": "$idade"}
    }},
    {"$project": {
        "_id": 0,
        "curso": "$_id",
        "media": {"$round": ["$media", 1]},
        "idade_media_meses": {"$multiply": [{"$round": ["$media", 1]}, 12]}
    }},
    {"$sort": {"idade_media_meses": 1}}
]):
    print(curso)

{'curso': 'Medicina', 'media': 19.5, 'idade_media_meses': 234.0}
{'curso': 'Computação', 'media': 21.5, 'idade_media_meses': 258.0}
{'curso': 'Ciência de Dados', 'media': 23.0, 'idade_media_meses': 276.0}
{'curso': 'Engenharia', 'media': 26.0, 'idade_media_meses': 312.0}


**Próximo:** `03_Conexao_Segura_e_App_Streamlit.ipynb`